In [1]:
import os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
import glob
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation
from torch.nn.functional import interpolate, softmax


/home/user1/charlie/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------------
# Dataset and DataLoader Setup
# -------------------------------
class SegFormerDataset(Dataset):
    def __init__(self, image_folder, feature_extractor):
        self.images = sorted([
            p for p in glob.glob(os.path.join(image_folder, '*.tif'))
            if 'mask' not in os.path.basename(p)
        ])
        self.masks = [
            os.path.join(image_folder, os.path.basename(p).replace('.tif','_mask.tif'))
            for p in self.images
        ]
        for m in self.masks:
            if not os.path.exists(m):
                raise FileNotFoundError(f"Mask not found: {m}")
        self.fe   = feature_extractor
        self.size = tuple(self.fe.size.values())

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB").resize(self.size)
        msk = Image.open(self.masks[idx]).convert("L").resize(self.size)
        pixel_values = self.fe(images=img, return_tensors="pt").pixel_values.squeeze(0)
        labels = torch.from_numpy(np.array(msk) > 127).long()
        return pixel_values, labels

def get_loaders(folder, batch_size=4, num_workers=4):
    test_ids = {44, 45, 46, 47}
    val_ids  = {36, 37, 38, 39, 40, 41, 42, 43}
    fe = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
    ds = SegFormerDataset(folder, fe)

    train_idx, val_idx, test_idx = [], [], []
    for i, img_path in enumerate(ds.images):
        pid = int(os.path.basename(img_path).split('_')[0])
        if pid in test_ids:
            test_idx.append(i)
        elif pid in val_ids:
            val_idx.append(i)
        else:
            train_idx.append(i)

    return {
        'train': DataLoader(Subset(ds, train_idx), batch_size=batch_size, shuffle=True, num_workers=num_workers),
        'val':   DataLoader(Subset(ds, val_idx),   batch_size=batch_size, shuffle=False, num_workers=num_workers),
        'test':  DataLoader(Subset(ds, test_idx),  batch_size=batch_size, shuffle=False, num_workers=num_workers),
    }

# -------------------------------
# Inference and Evaluation
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model(model_name):
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b0-finetuned-ade-512-512",
        num_labels=2, ignore_mismatched_sizes=True
    )
    model.load_state_dict(torch.load(f"best_segformer_{model_name}_0504.pt", map_location=device))
    return model.to(device).eval()

def save_mask(tensor, path):
    mask = (tensor.squeeze().cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(mask).save(path)

def run_inference(model_name, test_name, test_loader, output_dir):
    model = load_model(model_name)
    os.makedirs(output_dir, exist_ok=True)

    dice_scores = []
    file_names = []
    all_inputs = []
    all_gts = []
    fg_ratios = []

    with torch.no_grad():
        for idx, (px, lbl) in enumerate(tqdm(test_loader, desc=f"{model_name} on {test_name}")):
            px, lbl = px.to(device), lbl.to(device)
            output = model(pixel_values=px)
            logits = interpolate(output.logits, size=lbl.shape[-2:], mode="bilinear", align_corners=False)
            probs = softmax(logits, dim=1)[:, 1]
            preds = (probs > 0.5).float()

            for b in range(preds.size(0)):
                index = test_loader.dataset.indices[idx * px.size(0) + b]
                file = os.path.basename(test_loader.dataset.dataset.images[index])
                pred_path = os.path.join(output_dir, file.replace('.tif', '_pred.tif'))
                save_mask(preds[b], pred_path)

                intersection = (preds[b] * lbl[b]).sum()
                pred_sum = preds[b].sum()
                gt_sum = lbl[b].sum()
                total_pixels = preds[b].numel()
                fg_ratio = (preds[b].sum() + lbl[b].sum()) / total_pixels

                if fg_ratio < 0.001:
                    dice = 1.0  # trivial match
                else:
                    dice = (2 * (preds[b] * lbl[b]).sum()) / (preds[b].sum() + lbl[b].sum() + 1e-6)

                dice_scores.append(dice.item() if isinstance(dice, torch.Tensor) else dice)
                file_names.append(file)
                all_inputs.append(test_loader.dataset.dataset.images[index])
                all_gts.append(test_loader.dataset.dataset.masks[index])
                fg_ratios.append(fg_ratio.item() if isinstance(fg_ratio, torch.Tensor) else fg_ratio)



    return file_names, dice_scores, all_inputs, all_gts, fg_ratios

def plot_best_worst(file_names, dice_scores, all_inputs, all_gts, fg_ratios, output_dir, label, summary_dir):
    os.makedirs(summary_dir, exist_ok=True)
    
    # --- 1. compute predicted & GT foreground ratios (fraction of non-zero pixels) ---
    fg_ratios_pred = [
        np.array(Image.open(os.path.join(output_dir, f.replace('.tif', '_pred.tif')))).sum() / 255.0 / (512 * 512)
        for f in file_names
    ]
    fg_ratios_gt = [
        np.array(Image.open(gt)).sum() / 255.0 / (512 * 512)
        for gt in all_gts
    ]

    # --- 2. filter best candidates: both pred & GT must exceed threshold ---
    min_fg = 0.001  # 0.1% of image
    best_candidates = [
        (i, d) 
        for i, (d, fg_p, fg_g) in enumerate(zip(dice_scores, fg_ratios_pred, fg_ratios_gt))
        if fg_p > min_fg and fg_g > min_fg
    ]
    
    if best_candidates:
        best_idx = max(best_candidates, key=lambda x: x[1])[0]
    else:
        best_idx = int(np.argmax(dice_scores))

    # filter out zero dice scores for worst selection
    non_zero_dice = [(i, d) for i, d in enumerate(dice_scores) if d > 0]
    if non_zero_dice:
        worst_idx = min(non_zero_dice, key=lambda x: x[1])[0]
    else:
        worst_idx = int(np.argmin(dice_scores))  # fallback if all are zero


    # --- 3. plot both best & worst in turn ---
    for kind, idx in [("best", best_idx), ("worst", worst_idx)]:
        img_path, gt_path = all_inputs[idx], all_gts[idx]
        pred_path = os.path.join(output_dir, file_names[idx].replace('.tif', '_pred.tif'))
        
        img  = Image.open(img_path).convert("L")
        gt   = Image.open(gt_path).convert("L")
        pred = Image.open(pred_path).convert("L")

        # create a fixed-aspect subplot
        fig, axs = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
        for ax, im, title in zip(
            axs,
            [img, gt, pred],
            ["Input", "Ground Truth", "Prediction"]
        ):
            ax.imshow(im, cmap="gray")
            ax.set_title(title)
            ax.axis("off")
            ax.set_aspect('equal')  # ensure each panel is square/rectangular

        # overlay label on the rightmost panel
        axs[2].text(
            0, 10,
            f"{kind.capitalize()} prediction",
            fontsize=12,
            color="white",
            backgroundcolor="purple",
            fontweight="bold"
        )

        plt.suptitle(f"{label.upper()} {kind.upper()} {file_names[idx]}  Dice={dice_scores[idx]:.3f}")
        plt.savefig(os.path.join(summary_dir, f"{label}_{kind}.png"))
        plt.close()



In [3]:
# -------------------------------
# Main Loop
# -------------------------------
base_path = "/home/user1/charlie/code/deep_learning/project"
summary_dir = os.path.join(base_path, "best_worst")
experiments = {
    "raw":    os.path.join(base_path, "ultrasound-nerve-segmentation/train"),
    "filter": os.path.join(base_path, "ultrasound-nerve-segmentation_2Dfilter/train"),
    "clahe":  os.path.join(base_path, "ultrasound-nerve-segmentation_clahe/train"),
}

for model_name in experiments:
    for test_name in experiments:
        test_folder = experiments[test_name]
        output_dir = os.path.join(test_folder, f"test_mask_{model_name}")
        loaders = get_loaders(test_folder, batch_size=4)
        test_loader = loaders['test']
        fnames, dvals, inputs, gts, fg_ratios = run_inference(model_name, test_name, test_loader, output_dir)

        plot_best_worst(fnames, dvals, inputs, gts, fg_ratios, output_dir, f"{model_name}_on_{test_name}", summary_dir)


print(f"✅ All PNGs saved to: {summary_dir}")

/home/user1/charlie/.conda/lib/python3.11/site-packages/transformers/models/segformer/feature_extraction_segformer.py:28: FutureWarning: The class SegformerFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use SegformerImageProcessor instead.
  warnings.warn(
/home/user1/charlie/.conda/lib/python3.11/site-packages/transformers/utils/deprecation.py:165: UserWarning: The following named arguments are not valid for `SegformerFeatureExtractor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([2]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([2

✅ All PNGs saved to: /home/user1/charlie/code/deep_learning/project/best_worst
